In [1]:
import pandas as pd 
import urllib.request
import requests
import tqdm
from pathlib import Path
import numpy as np

cpcb_site_json = "../config/cpcb_sites.json"
cpcb_site_df = pd.read_json(cpcb_site_json)

files = {}
labels = {}

for city in cpcb_site_df["dropdown"]["stations"]:
    for site in cpcb_site_df["dropdown"]["stations"][city]:
        site_substring = site["label"].replace(",","").replace("-","").split()
        file = f"{site["value"]}_{"_".join(site_substring)}_15Min.csv"
        file_name = f"CPCB_{site_substring[0]}{site_substring[1]}_2025.csv"
        files.update({file_name : file})
        labels.update({site["label"] : f"{site_substring[0]}{site_substring[1]}"})

labels

{'Evelyn Lodge, Asansol - WBPCB': 'EvelynLodge',
 'Mahabir Colliery, Asansol - WBPCB': 'MahabirColliery',
 'Trivenidevi Bhalotia College, Asansol - WBPCB': 'TrivenideviBhalotia',
 'Asansol Court Area, Asansol - WBPCB': 'AsansolCourt',
 'Railway Colony, Barmer - RSPCB': 'RailwayColony',
 'Bandra Kurla Complex, Mumbai - MPCB': 'BandraKurla',
 'Deonar, Mumbai - IITM': 'DeonarMumbai',
 'Colaba, Mumbai - MPCB': 'ColabaMumbai',
 'Borivali East, Mumbai - MPCB': 'BorivaliEast',
 'Vile Parle West, Mumbai - MPCB': 'VileParle',
 'Shivaji Nagar, Mumbai - BMC': 'ShivajiNagar',
 'Chembur, Mumbai - MPCB': 'ChemburMumbai',
 'Kandivali West, Mumbai - BMC': 'KandivaliWest',
 'Kandivali East, Mumbai - MPCB': 'KandivaliEast',
 'Khindipada-Bhandup West, Mumbai - IITM': 'KhindipadaBhandupWest',
 'Borivali East, Mumbai - IITM': 'BorivaliEast',
 'Bandra Kurla Complex, Mumbai - IITM': 'BandraKurla',
 'Chhatrapati Shivaji Intl. Airport (T2), Mumbai - MPCB': 'ChhatrapatiShivaji',
 'Sion, Mumbai - MPCB': 'SionMum

In [2]:
LocFile = "../data/raw/misc/26May26AqiData.csv"
StationFile = "../config/Stations.csv"

df = pd.read_csv(LocFile)

def get_coords(station):

    result = df[df["station"] == station][["latitude", "longitude"]]
    if not result.empty:
        return np.round(result.iloc[0]["latitude"], 2), np.round(result.iloc[0]["longitude"], 2)
    else:
        return None, None

locations = []

for station, file_name in labels.items():
    lat, lon = get_coords(station)
    
     
    if lat is None or lon is None:
        print(f"Coordinates not found for station: {station}. Skipping.")
        continue
    
    locations.append({
        "Station_Name" : file_name,
        "Latitude_Start" : lat - 0.1,
        "Latitude_End" : lat + 0.1,
        "Longitude_Start" : lon - 0.1,
        "Longitude_End" : lon + 0.1,
        "latitude" : lat,
        "longitude" : lon
    })

locations_df = pd.DataFrame(locations)
master_Stat_df = pd.concat([pd.read_csv(StationFile), locations_df])
master_Stat_df["Station_id"] = range(1, len(master_Stat_df) + 1)
master_Stat_df.to_csv("../config/Station_Locations.csv", index=False)


Coordinates not found for station: Vile Parle West, Mumbai - MPCB. Skipping.
Coordinates not found for station: Kandivali East, Mumbai - MPCB. Skipping.
Coordinates not found for station: Bandra, Mumbai - MPCB. Skipping.
Coordinates not found for station: Vasai West, Mumbai - MPCB. Skipping.
Coordinates not found for station: Mulund West, Mumbai - MPCB. Skipping.
Coordinates not found for station: More Chowk Waluj, Aurangabad - MPCB. Skipping.
Coordinates not found for station: Gurdeo Nagar, Aurangabad - BSPCB. Skipping.
Coordinates not found for station: Ved Vihar-Loni, Ghaziabad - UPPCB. Skipping.
Coordinates not found for station: MIET College, Meerut - UPPCB. Skipping.
Coordinates not found for station: IITK, Kanpur - IITK. Skipping.
Coordinates not found for station: RVCE-Mailasandra, Bengaluru - KSPCB. Skipping.
Coordinates not found for station: Kasturi Nagar, Bengaluru - KSPCB. Skipping.
Coordinates not found for station: Sanegurava Halli, Bengaluru - KSPCB. Skipping.
Coordinat

In [30]:
df = pd.read_csv("../config/Stations.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Station_id       6 non-null      int64  
 1   Station_Name     6 non-null      str    
 2   Latitude_start   6 non-null      float64
 3   Latitude_End     6 non-null      float64
 4   Longitude_Start  6 non-null      float64
 5   Longitude_End    6 non-null      float64
dtypes: float64(4), int64(1), str(1)
memory usage: 420.0 bytes


In [9]:
baseurl = "https://airquality.cpcb.gov.in/dataRepository/download_file?file_name=Raw_data/15Min/2025/"
for file_name, file in tqdm.tqdm(files.items()):
    # print(f"{baseurl}{file}")
    if Path(f"../data/raw/PM_CPCB/{file_name}").exists():
        print(f"{file_name} already exists. Skipping download.")
        continue
    try:
        urllib.request.urlretrieve(f"{baseurl}{file}", f"../data/raw/PM_CPCB/{file_name}")
    except Exception as e:
        print(f"Error occurred while downloading {file_name}: {e}")

  0%|          | 0/571 [00:00<?, ?it/s]

CPCB_EvelynLodge_2025.csv already exists. Skipping download.
CPCB_MahabirColliery_2025.csv already exists. Skipping download.
CPCB_TrivenideviBhalotia_2025.csv already exists. Skipping download.
CPCB_AsansolCourt_2025.csv already exists. Skipping download.
CPCB_RailwayColony_2025.csv already exists. Skipping download.
CPCB_BandraKurla_2025.csv already exists. Skipping download.
CPCB_DeonarMumbai_2025.csv already exists. Skipping download.
CPCB_ColabaMumbai_2025.csv already exists. Skipping download.
CPCB_BorivaliEast_2025.csv already exists. Skipping download.
CPCB_VileParle_2025.csv already exists. Skipping download.
CPCB_ShivajiNagar_2025.csv already exists. Skipping download.
CPCB_ChemburMumbai_2025.csv already exists. Skipping download.
CPCB_KandivaliWest_2025.csv already exists. Skipping download.
CPCB_KandivaliEast_2025.csv already exists. Skipping download.


  3%|▎         | 15/571 [00:00<00:17, 32.45it/s]

Error occurred while downloading CPCB_KhindipadaBhandupWest_2025.csv: HTTP Error 500: Internal Server Error
CPCB_ChhatrapatiShivaji_2025.csv already exists. Skipping download.
CPCB_SionMumbai_2025.csv already exists. Skipping download.
CPCB_BandraMumbai_2025.csv already exists. Skipping download.
CPCB_VasaiWest_2025.csv already exists. Skipping download.
CPCB_GhatkoparMumbai_2025.csv already exists. Skipping download.
CPCB_KurlaMumbai_2025.csv already exists. Skipping download.


  4%|▍         | 22/571 [00:00<00:19, 28.67it/s]

...

Error occurred while downloading CPCB_IndiraNagar_2025.csv: HTTP Error 500: Internal Server Error


100%|█████████▉| 570/571 [28:47<00:01,  1.55s/it]

Error occurred while downloading CPCB_VidhyanagarBhavnagar_2025.csv: HTTP Error 500: Internal Server Error


100%|██████████| 571/571 [28:47<00:00,  3.02s/it]

Error occurred while downloading CPCB_BirBeed_2025.csv: HTTP Error 500: Internal Server Error


In [38]:
df = pd.read_csv("../config/Station_Locations.csv")
df["Station_Name"].nunique()

485